In [ ]:
"""Task E: Efficiency Profile - Parameters, Time, Memory"""

import torch
import time
import pandas as pd
from models.baseline import GRUForecast
from models.igru_variant import iGRUForecast

def profile_model(model, model_name, input_shape=(32, 96, 7)):
    """Profile a single model for efficiency metrics"""
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    # 1. Parameter count
    param_count = sum(p.numel() for p in model.parameters())
    
    # 2. Training time per epoch (simulated)
    # We'll use a dummy forward-backward pass
    x = torch.randn(*input_shape).to(device)
    y = torch.randn(32, 24, 1).to(device)
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Warm-up
    for _ in range(5):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
    
    # Measure training time per batch
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    n_iterations = 50
    
    for _ in range(n_iterations):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    training_time_per_batch = (time.time() - start) / n_iterations
    
    # 3. Inference time
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    n_inference = 200
    
    with torch.no_grad():
        for _ in range(n_inference):
            out = model(x)
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    inference_time_per_batch = (time.time() - start) / n_inference
    
    # 4. Memory usage (if CUDA available)
    memory_usage = 0
    if torch.cuda.is_available():
        memory_usage = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
        torch.cuda.reset_peak_memory_stats()
    
    return {
        'Model': model_name,
        'Parameters': param_count,
        'Train Time/Batch (ms)': training_time_per_batch * 1000,
        'Inference Time/Batch (ms)': inference_time_per_batch * 1000,
        'Peak Memory (MB)': memory_usage
    }

def main():
    """Profile both models and create comparison table"""
    
    print("="*70)
    print("EFFICIENCY PROFILE")
    print("="*70)
    
    # Define models
    input_dim = 7
    models = {
        'GRU': GRUForecast(input_dim, hidden_dim=64, horizon=24, out_dim=1, num_layers=2),
        'iGRU': iGRUForecast(input_dim, hidden_dim=64, horizon=24, out_dim=1, num_layers=2)
    }
    
    results = []
    for name, model in models.items():
        print(f"\nProfiling {name}...")
        profile = profile_model(model, name)
        results.append(profile)
        print(f"  Parameters: {profile['Parameters']:,}")
        print(f"  Train Time/Batch: {profile['Train Time/Batch (ms)']:.3f} ms")
        print(f"  Inference Time/Batch: {profile['Inference Time/Batch (ms)']:.3f} ms")
    
    # Create comparison DataFrame
    df = pd.DataFrame(results)
    df = df.set_index('Model')
    
    print("\n" + "="*70)
    print("EFFICIENCY COMPARISON TABLE")
    print("="*70)
    print(df.to_string())
    
    # Save to CSV
    df.to_csv('results/efficiency_comparison.csv')
    print("\n✅ Results saved to results/efficiency_comparison.csv")
    
    return df

if __name__ == "__main__":
    df = main()